# GNN Results Analysis

Import packages

In [37]:
import json
import pandas as pd
from pathlib import Path
from importlib.metadata import version

# Versions used pandas==2.3.3 numpy==2.2.6
for package in ["pandas", "numpy"]:
    print(f"{package}: {version(package)}")

pandas: 2.3.3
numpy: 2.2.6


Load result files

In [38]:
result_file_names = [
    'compare_gnn_graphs_d3_r5_p001_vs_mwpm_d3_r5_p001.json',
    'compare_gnn_graphs_d3_r5_p0001_vs_mwpm_d3_r5_p0001.json',
    'compare_gnn_graphs_d3_r5_p0005_vs_mwpm_d3_r5_p0005.json',
    'compare_gnn_graphs_d3_r5_p00005_vs_mwpm_d3_r5_p00005.json',
    'compare_gnn_graphs_d5_r5_p0005_vs_mwpm_d5_r5_p0005.json',
    'compare_gnn_graphs_d7_r5_p0005_vs_mwpm_d7_r5_p0005.json',
]

# Project root (assuming notebook runs from analysis/)
project_root = Path('..')

# Construct file paths
metrics_dir = project_root / 'gnn_qec-main' / 'outputs' / 'metrics'

result_file_paths = [
    metrics_dir / file_name
    for file_name in result_file_names
]

# Load JSON results
results = []

for result_file_path in result_file_paths:
    with open(result_file_path, 'r') as f:
        results.append(json.load(f))

Calculate LEP, LER and report results in table.

In [39]:
def calculate_ler(lep: float, r: int) -> float:
    """
    Calculate LER through direct inversion, assumes constant amount of rounds for all shots.

    :param lep: Logical error probability (1 - accuracy).
    :param r: Number of rounds.
    :return: Logical error rate per round.
    """
    return 0.5 * (1 - ((1 - 2 * lep) ** (1.0 / r)))

table_rows = []
for result in results:
    distance = result['distance']
    rounds = result['rounds']
    p = result['p']

    # GNN metrics path already includes outputs/metrics/
    gnn_metric_path = (
        project_root
        / 'gnn_qec-main'
        / result['gnn_metrics_path']
    )

    # Read GNN metrics
    with open(gnn_metric_path, 'r') as f:
        gnn_metrics = json.load(f)

    test_shots = gnn_metrics['test']['num_examples']
    trivial_error_rate = result['positive_label_fraction']

    gnn_lep = 1 - result['gnn_accuracy']
    mwpm_lep = 1 - result['mwpm_accuracy']
    gnn_ler = calculate_ler(gnn_lep, rounds)
    mwpm_ler = calculate_ler(mwpm_lep, rounds)
    gnn_gain = trivial_error_rate / gnn_lep
    mwpm_gain = trivial_error_rate / mwpm_lep

    # Add one row per result
    table_rows.append({
        'd': distance,
        'p': p,
        'Test shots': test_shots,
        'Trivial error rate': trivial_error_rate,
        'GNN LEP': gnn_lep,
        'MWPM LEP': mwpm_lep,
        'GNN gain': gnn_gain,
        'MWPM gain': mwpm_gain,
        'GNN LER': gnn_ler,
        'MWPM LER': mwpm_ler,
    })

# Create DataFrame
results_df = pd.DataFrame(table_rows)

def format_p(value):
    mantissa, exponent = f"{float(value):.1e}".split("e")
    mantissa = f"{float(mantissa):g}"
    exponent = int(exponent)

    return f"{mantissa} × 10<sup>{exponent}</sup>"

# Ensure d and p are numeric
results_df['d'] = pd.to_numeric(results_df['d'])
results_df['p'] = pd.to_numeric(results_df['p'])

# Sort first by distance, then by p
results_df = (
    results_df
    .sort_values(by=['d', 'p'], ascending=[True, True])
    .reset_index(drop=True)
)

# Display formatted table
display(
    results_df.style.format({
        'p': format_p,
        'Test shots': '{:,}',
        'Trivial error rate': '{:.6f}',
        'GNN LEP': '{:.6f}',
        'MWPM LEP': '{:.6f}',
        'GNN gain': '{:.4f}',
        'MWPM gain': '{:.4f}',
        'GNN LER': '{:.6f}',
        'MWPM LER': '{:.6f}',
    })
)

,d,p,Test shots,Trivial error rate,GNN LEP,MWPM LEP,GNN gain,MWPM gain,GNN LER,MWPM LER
0,3,5 × 10-4,"7,500",0.019600,0.004000,0.000400,4.9000,49.0000,0.000803,0.000080
1,3,1 × 10-3,"7,500",0.035333,0.009733,0.001867,3.6301,18.9286,0.001962,0.000374
2,3,5 × 10-3,"15,000",0.144333,0.063667,0.026733,2.2670,5.3990,0.013436,0.005465
3,3,1 × 10-2,"7,500",0.246933,0.187200,0.092267,1.3191,2.6763,0.044772,0.019989
4,5,5 × 10-3,"7,500",0.229467,0.225467,0.016933,1.0177,13.5512,0.056499,0.003433
5,7,5 × 10-3,"15,000",0.287000,0.287133,0.007267,0.9995,39.4954,0.078500,0.001462
